In [1]:
# Cell 1 — Imports
import sys
import os

sys.path.append(os.path.abspath("../src"))

import random
import torch
from torch import nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset
from collections import defaultdict
from typing import List, Dict, Tuple, Optional, Type, Union

from abstractions3d.primitives.shapes import Box3D
from abstractions3d.primitives.visualization import visualize_boxes_3d
from abstractions3d.dsl.core import Shape3D
from abstractions3d.dsl.nodes import Rect3D, Move3D, Union3D, SymTrans3D, SymRef3D
from abstractions3d.dsl.instantiation import instantiate_3d
from abstractions3d.data.blueprint import Table3D, Chair3D, Bench3D


In [2]:
# Cell 2 — Dataset generation
def sample_uniform(low: float, high: float) -> float:
    return random.uniform(low, high)

def generate_dataset(num_samples: int = 50) -> List[Shape3D]:
    dataset = []
    for _ in range(num_samples):
        # Tables
        table = Table3D(
            top_length=sample_uniform(3.0, 6.0),
            top_depth=sample_uniform(1.5, 3.0),
            top_thickness=sample_uniform(0.1, 0.3),
            leg_length=sample_uniform(0.2, 0.4),
            leg_depth=sample_uniform(0.2, 0.4),
            leg_height=sample_uniform(1.0, 2.0)
        )
        dataset.append(table)

        # Chairs
        chair = Chair3D(
            seat_length=sample_uniform(1.0, 2.0),
            seat_depth=sample_uniform(1.0, 2.0),
            seat_thickness=sample_uniform(0.2, 0.5),
            leg_length=sample_uniform(0.1, 0.3),
            leg_depth=sample_uniform(0.1, 0.3),
            leg_height=sample_uniform(0.5, 1.0),
            backrest_height=sample_uniform(1.0, 2.0),
            backrest_thickness=sample_uniform(0.1, 0.3)
        )
        dataset.append(chair)

        # Benches
        bench = Bench3D(
            seat_length=sample_uniform(2.0, 5.0),
            seat_depth=sample_uniform(0.5, 1.0),
            seat_thickness=sample_uniform(0.2, 0.5),
            leg_length=sample_uniform(0.1, 0.3),
            leg_depth=sample_uniform(0.1, 0.3),
            leg_height=sample_uniform(0.5, 1.0),
            backrest_height=sample_uniform(0.5, 1.5),
            backrest_thickness=sample_uniform(0.1, 0.3)
        )
        dataset.append(bench)
    return dataset

In [3]:
# Cell 3 — Structure extraction helpers
def add_dicts(d1: Dict[str, List], d2: Dict[str, List]) -> Dict[str, List]:
    for k in d2:
        d1[k].extend(d2[k])
    return d1

def get_singletons(shapes: Union[Shape3D,List[Shape3D]]) -> Dict[str,List[Tuple[float,...]]]:
    singletons = defaultdict(list)
    if isinstance(shapes,list):
        for s in shapes:
            singletons = add_dicts(singletons,get_singletons(s))
        return singletons
    t, params = shapes.param_tuple()
    floats = tuple(p for p in params if isinstance(p,(float,int)))
    if floats:
        singletons[t.__name__].append(floats)
    for c in getattr(shapes,'children',[]):
        singletons = add_dicts(singletons,get_singletons(c))
    return singletons

def get_pairs(shapes: Union[Shape3D,List[Shape3D]]) -> Dict[str,List[Tuple[float,...]]]:
    pairs = defaultdict(list)
    if isinstance(shapes,list):
        for s in shapes:
            pairs = add_dicts(pairs,get_pairs(s))
        return pairs
    if isinstance(shapes,Union3D):
        t, (c1,c2) = shapes.param_tuple()
        t1, p1 = c1.param_tuple()
        t2, p2 = c2.param_tuple()
        key = f"{t.__name__}({t1.__name__},{t2.__name__})"
        pairs[key].append(tuple([p for p in p1 if isinstance(p,(float,int))] +
                                [p for p in p2 if isinstance(p,(float,int))]))
        for c in shapes.children:
            pairs = add_dicts(pairs,get_pairs(c))
    else:
        for c in getattr(shapes,'children',[]):
            pairs = add_dicts(pairs,get_pairs(c))
    return pairs

In [4]:
# Cell 4 — Autoencoder definition
class Autoencoder(nn.Module):
    def __init__(self,input_dim:int,latent_dim:int):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim,64), nn.ReLU(), nn.Linear(64,latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim,64), nn.ReLU(), nn.Linear(64,input_dim))
    def forward(self,x:torch.Tensor):
        z = self.encoder(x)
        return z,self.decoder(z)

In [5]:
# Cell 5 — Find abstractions
def find_abstractions(structures: Dict[str,List[Tuple[float,...]]],
                      retrain_iterations: int = 10,
                      error_threshold: float = 0.05) -> Tuple[Dict[str, nn.Module], Dict[str,List[float]]]:
    models = {}
    losses = {}
    for name, params in structures.items():
        if not params: continue
        float_params = [list(p) for p in params if any(isinstance(x,(float,int)) for x in p)]
        if not float_params: continue
        num_float = len(float_params[0])
        if num_float < 2: continue

        train_tensor = torch.tensor(float_params,dtype=torch.float32)
        train_dl = DataLoader(TensorDataset(train_tensor), batch_size=32, shuffle=True)
        model = Autoencoder(input_dim=num_float, latent_dim=max(1,num_float-1))
        optimizer = AdamW(model.parameters(), lr=1e-3)
        loss_fn = lambda pred,target: torch.mean(torch.max(torch.abs(pred-target),dim=-1)[0])

        model.train()
        losses[name] = []
        for _ in range(retrain_iterations):
            batch_losses = []
            for batch in train_dl:
                x = batch[0]
                optimizer.zero_grad()
                _, x_rec = model(x)
                loss = loss_fn(x_rec,x)
                loss.backward()
                optimizer.step()
                batch_losses.append(loss.item())
            losses[name].append(sum(batch_losses)/len(batch_losses))
        models[name] = model
        print(f"Trained {name}, final loss: {losses[name][-1]:.4f}")
    return models, losses

In [6]:
# Cell 6 — Integrate abstractions
def integrate_abstractions(shape: Shape3D, models: Dict[str, nn.Module],
                           error_threshold: float = 0.05,
                           stats: Optional[Dict[str,int]] = None) -> Shape3D:
    if stats is None: stats = {'replaced':0}

    # Composite
    if isinstance(shape,Union3D):
        new_c1 = integrate_abstractions(shape.children[0],models,error_threshold,stats)
        new_c2 = integrate_abstractions(shape.children[1],models,error_threshold,stats)
        type_str = f"Union3D({type(new_c1).__name__},{type(new_c2).__name__})"
        float_params = [p for p in new_c1.param_tuple()[1] if isinstance(p,(float,int))] + \
                       [p for p in new_c2.param_tuple()[1] if isinstance(p,(float,int))]
        if type_str in models and float_params:
            model = models[type_str]
            input_tensor = torch.tensor(float_params)[None,:]
            with torch.no_grad():
                _, recon = model(input_tensor)
                error = torch.max(torch.abs(recon - input_tensor)).item()
            if error < error_threshold:
                stats['replaced'] += 1
                return AbstractionNode([Union3D,type(new_c1),type(new_c2)],
                                       torch.tensor(float_params,dtype=torch.float32),
                                       model)
        return Union3D(new_c1,new_c2)

    # Singleton
    type_, params = shape.param_tuple()
    float_params = [p for p in params if isinstance(p,(float,int))]
    type_str = type_.__name__
    if type_str in models and float_params:
        model = models[type_str]
        input_tensor = torch.tensor(float_params)[None,:]
        with torch.no_grad():
            _, recon = model(input_tensor)
            error = torch.max(torch.abs(recon - input_tensor)).item()
        if error < error_threshold:
            stats['replaced'] += 1
            return AbstractionNode([type_], torch.tensor(float_params,dtype=torch.float32), model)

    new_children = [integrate_abstractions(c,models,error_threshold,stats) 
                    for c in getattr(shape,'children',[])]
    shape.children = new_children
    return shape